## 파이썬 원-핫 인코딩(One-hot encoding)
 - 참조: http://wikidocs.net/22647
 - 한 개 요소는 True, 나머지 요소는 False를 만드는 기법
 - 원-핫 인코딩이 필요한 이유
  * scikit-learn에서 제공하는 머신 러닝 알고리즘은 문자열 값을 입력값으로 허용하지 않음
  * 문자열 값을 숫자형 자료로 인코딩하는 전처리 작업이 필요
 - scikit-learn에서 제공하는 머신러닝 알고리즘에 필요한 데이터 구성
  * 모든 데이터는 숫자형으로만 구성되어야 한다
  * 데이터에 빈 값이 없어야 한다

In [ ]:
import pandas as pd

train = pd.DataFrame({'num1': [1,2,3,4,5],
                      'num2': [10,20,30,40,50],
                      'cat1':['a','b','a','c','c']})

In [ ]:
train

,num1,num2,cat1
0,1,10,a
1,2,20,b
2,3,30,a
3,4,40,c
4,5,50,c


In [ ]:
# 데이터를 레이블화 시키는 방법
cat_lst=train.cat1.unique() # 고유값 추출
cat_num = [i for i in enumerate(cat_lst)] # 벡터 값에 대한 레이블(번호)
print(cat_lst, cat_num)

['a' 'b' 'c'] [(0, 'a'), (1, 'b'), (2, 'c')]


In [ ]:
# 원 핫 인코딩 (이것은 손 코딩 하는 방법)

one_h_lst = [] # 전체 데이터

for f in train.cat1:
  lst = [] # 각 행 단위의 데이터

  for i in cat_lst: # 트레이닝 데이터의 cat1의 고유값 범위 내에서 돌린다

    if i  == f:
      lst.append(1) # 각 행의 레이블하고 같으면 1을 넣고

    else:
      lst.append(0) # 각 행의 레이블하고 다르면 0을 넣는다.

  one_h_lst.append(lst) # lst 리스트 뽑아낸 것을 one_h_lst에 다시 넣는다.

print(one_h_lst)



[[1, 0, 0], [0, 1, 0], [1, 0, 0], [0, 0, 1], [0, 0, 1]]


In [ ]:
train1 = pd.concat([train, pd.DataFrame(one_h_lst, columns=cat_lst)],axis=1)
# 기존 train 데이터프레임에 one_h_lst 작업한 것을 합친다

train1

,num1,num2,cat1,a,b,c
0,1,10,a,1,0,0
1,2,20,b,0,1,0
2,3,30,a,1,0,0
3,4,40,c,0,0,1
4,5,50,c,0,0,1


In [ ]:
train_1 = pd.get_dummies(train, columns=['cat1'], prefix = 'cat1')
# 기존의 cat1 열을 없애면서 각각의 필드값으로 영향을 준다
train_1

,num1,num2,cat1_a,cat1_b,cat1_c
0,1,10,1,0,0
1,2,20,0,1,0
2,3,30,1,0,0
3,4,40,0,0,1
4,5,50,0,0,1


pandas 모듈 사용

In [ ]:
# true / false

cat_lst=train['cat1'].value_counts().index

for c in cat_lst:
  train[f'cat1_{c}'] = train['cat1'] == c

train

,num1,num2,cat1,cat1_a,cat1_b,cat1_c
0,1,10,a,True,False,False
1,2,20,b,False,True,False
2,3,30,a,True,False,False
3,4,40,c,False,False,True
4,5,50,c,False,False,True


sklearn-OneHotEncoder 이용

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
ohe = OneHotEncoder(sparse=False)
train_cat = ohe.fit_transform(train[['cat1']])
print(train_cat)

[[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 0. 1.]]


In [ ]:
ohe.categories_

[array(['a', 'b', 'c'], dtype=object)]

In [ ]:
pd.DataFrame(train_cat, columns=['cat1_' + cal for cal in ohe.categories_])

,cat1_a,cat1_b,cat1_c
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0


In [ ]:
# train.drop(columns='cat1')
# train.drop('cat1', axis=1)
pd.concat([train.drop(columns='cat1'),
           pd.DataFrame(train_cat, columns=['cat1_' + cal for cal in ohe.categories_[0]])],
           axis=1)

,num1,num2,cat1_a,cat1_b,cat1_c,cat1_a,cat1_b,cat1_c
0,1,10,True,False,False,1.0,0.0,0.0
1,2,20,False,True,False,0.0,1.0,0.0
2,3,30,True,False,False,1.0,0.0,0.0
3,4,40,False,False,True,0.0,0.0,1.0
4,5,50,False,False,True,0.0,0.0,1.0


### 성적표를 이용하여 원-핫 인코딩

In [ ]:
import pandas as pd
import numpy as np

# 데이터 읽어오기
df = pd.read_csv('성적표.csv', encoding='cp949')
df.head()

,순번,이름,학과,남/여,학년,이론,실기
0,1,송윤재,환경디자인원예학과,여자,2,NaN,NaN
1,2,강민형,사회복지학과,여자,3,NaN,NaN
2,3,강예린,환경디자인원예학과,여자,3,NaN,NaN
3,4,고보빈,경영학과,여자,4,NaN,NaN
4,5,김다정,보건관리학과,여자,3,NaN,NaN


In [ ]:
# 이론 / 실기 NaN 값을 0 ~ 100 사이의 임의 값으로 채우기
# np.random.randint(0,101,size=개수) => 0에서 100까지 임의의 값을 정한 갯수만큼 생성
df.이론 = np.random.randint(1,101,len(df))
df.실기 = np.random.randint(1,101,len(df))
df.head()

,순번,이름,학과,남/여,학년,이론,실기
0,1,송윤재,환경디자인원예학과,여자,2,20,60
1,2,강민형,사회복지학과,여자,3,36,48
2,3,강예린,환경디자인원예학과,여자,3,25,9
3,4,고보빈,경영학과,여자,4,88,30
4,5,김다정,보건관리학과,여자,3,19,72


In [ ]:
# 학과 레이블 인코딩 (sklearn 사용)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le_no = le.fit(df.학과).transform(df.학과) # 피팅 결과 값
#print(le_no)
#print(le.classes_) # 레이블 고유 값

df['학과코드'] = le_no
df.head()

[16  7 16  4  6  8 14 15 14  7  9  3  4 15  4  2  4 14  0 12 10 10  6 13
  3 13  4  5 12 16  3  1 12  1 11]
['IT융합공학과' '간호학과' '건축학과' '경영정보학과' '경영학과' '동물생명자원학과' '보건관리학과' '사회복지학과'
 '상담심리학과' '식품영양학과' '영어영문학전공' '영어통번역전공' '음악학과' '일본어과' '중국어과' '화학생명과학과'
 '환경디자인원예학과']


,순번,이름,학과,남/여,학년,이론,실기,학과코드
0,1,송윤재,환경디자인원예학과,여자,2,20,60,16
1,2,강민형,사회복지학과,여자,3,36,48,7
2,3,강예린,환경디자인원예학과,여자,3,25,9,16
3,4,고보빈,경영학과,여자,4,88,30,4
4,5,김다정,보건관리학과,여자,3,19,72,6


In [ ]:
pd.get_dummies(df['남/여']).head()

,남자,여자
0,0,1
1,0,1
2,0,1
3,0,1
4,0,1


In [ ]:
df=pd.concat([df.drop(columns='남/여'), pd.get_dummies(df['남/여'])],
           axis=1)
df.head()

,순번,이름,학과,학년,이론,실기,학과코드,남자,여자
0,1,송윤재,환경디자인원예학과,2,20,60,16,0,1
1,2,강민형,사회복지학과,3,36,48,7,0,1
2,3,강예린,환경디자인원예학과,3,25,9,16,0,1
3,4,고보빈,경영학과,4,88,30,4,0,1
4,5,김다정,보건관리학과,3,19,72,6,0,1


In [ ]:
df=pd.concat([df.drop(columns='학과'), pd.get_dummies(df['학과'])],
           axis=1)
df.head()

,순번,이름,학년,이론,실기,학과코드,남자,여자,IT융합공학과,간호학과,...,사회복지학과,상담심리학과,식품영양학과,영어영문학전공,영어통번역전공,음악학과,일본어과,중국어과,화학생명과학과,환경디자인원예학과
0,1,송윤재,2,20,60,16,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2,강민형,3,36,48,7,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
2,3,강예린,3,25,9,16,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
3,4,고보빈,4,88,30,4,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,김다정,3,19,72,6,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
